# Bikeablity debugging

This notebook is to produce a bikeablity value for the "demand" approach and a small sample of "random" approaches. It is used for debugging. We implement one type of bikeablity in this, specifically the LTS Level + Distance cutoff approach. This is where between any two node pairs, we accept demand being connected when a route between the pairs can be done on a LTS with a maximum of 2 and a LTS Distance of less than 3km.

Approach:
- load some results
- calculate bikeablity
- save bikeablity for plotting
- plot map of growth for demand
- plot map of growth for random when it betters demand

In [ ]:
LTS_Limit = 1 # set what the maximum LTS we can use is. Use either "1" or "2"

## setup

In [ ]:
import pickle
import os
import glob
import osmnx as ox
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
import igraph as ig
import pandas as pd

## load data

In [ ]:
# load the OD demand GPKG
od_gpkg_path = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\gateshead\current_ltn_scenario\gateshead_current_ltn_scenario_greedy_demand_weighted.gpkg"
od_demand = gpd.read_file(od_gpkg_path)

# load the base bikeable network
G_bikeable_edges = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\gateshead\current_ltn_scenario\gateshead_biketrackcarall.gpkg", layer="edges")
G_bikeable_nodes = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\gateshead\current_ltn_scenario\gateshead_biketrackcarall.gpkg", layer="nodes")
G_bikeable_edges = G_bikeable_edges.set_index(["u", "v", "key"])
G_bikeable_nodes = G_bikeable_nodes.set_index("osmid")
exit_points = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\exports_gpkg\gateshead\current_ltn_scenario\gateshead_exit_points.gpkg")
G_bikeable_nx = ox.graph_from_gdfs(G_bikeable_nodes, G_bikeable_edges)

# load the LTNs 
ltns = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\data\gateshead\current_ltn_scenario\scored_neighbourhoods_gateshead.gpkg")

## make LTS weighted network

In [ ]:
## set up G_base
G_base = G_bikeable_nx.copy() # this is out base network throughout

##  make sure LTN edges are flagged in G_base (so we can consider them as part of the biking network)
_, edges_gdf = ox.graph_to_gdfs(G_base)
if edges_gdf.crs != ltns.crs:
    ltns = ltns.to_crs(edges_gdf.crs)
edges_gdf = edges_gdf.drop(columns=['index_left', 'index_right'], errors='ignore')
ltns = ltns.drop(columns=['index_left', 'index_right'], errors='ignore')
# This finds every edge that touches or is inside an LTN polygon
edges_in_ltn = gpd.sjoin(edges_gdf, ltns, how='inner', predicate='intersects')
ltn_edge_indices = set(edges_in_ltn.index)
ltn_flags = {}
for u, v, k in G_base.edges(keys=True):
    # If the edge index is in our joined set, flag it True
    ltn_flags[(u, v, k)] = (u, v, k) in ltn_edge_indices
nx.set_edge_attributes(G_base, ltn_flags, name='ltn_flag')
print(f"Flagged {len(ltn_edge_indices)} edges as being inside LTNs.")


## give each edge an LTS class
lts_mapping = {
    "motorway": 4, "motorway_link": 4, "trunk": 4, "trunk_link": 4,
    "primary": 4, "primary_link": 4, "secondary": 4, "secondary_link": 4,
    "tertiary": 3, "tertiary_link": 3, "unclassified": 3,
    "residential": 2, "living_street": 2,
    "cycleway": 1, "track": 1, "path": 1, "bridleway": 1, "footway": 1, "pedestrian": 1}
# if unknown
DEFAULT_LTS = 4 

## find LTS distance for each edge
for u, v, key, data in G_base.edges(keys=True, data=True):
    is_ltn = data.get('ltn_flag')
    if is_ltn is True or str(is_ltn).lower() == 'true':
        lts_class = 1
    else:
        highway_type = data.get('highway')
        if isinstance(highway_type, list):
            highway_type = highway_type
        # Look up the LTS class, using the default if the type isn't found
        lts_class = lts_mapping.get(highway_type, DEFAULT_LTS)
    # Get the physical length
    edge_length = data.get('length', 0)
    # Assign the new attributes to the edge
    data['lts_class'] = lts_class
    data['lts_length'] = edge_length * lts_class



if G_base.is_directed():
    G_base.to_undirected()

G_LTS_2 = G_base.copy()
edges_to_remove = []
for u, v, key, data in G_LTS_2.edges(keys=True, data=True):
    if data.get('lts_class', 4) > LTS_Limit:
        edges_to_remove.append((u, v, key))
G_LTS_2.remove_edges_from(edges_to_remove)


# get G_base edges with lts for plotting later
_, G_base_edges_gdf = ox.graph_to_gdfs(G_base)

In [ ]:
# CONVERT G_LTS_2 TO IGRAPH
print("Converting NetworkX G_LTS_2 to igraph...")
# Create a dictionary mapping OSM IDs to igraph integer indices
osmid_to_idx = {n: i for i, n in enumerate(G_LTS_2.nodes())}

g_lts2_ig = ig.Graph(directed=False)
g_lts2_ig.add_vertices(len(G_LTS_2.nodes()))

# Extract edges and their LTS lengths
edges = []
weights = []
for u, v, data in G_LTS_2.edges(data=True):
    edges.append((osmid_to_idx[u], osmid_to_idx[v]))
    weights.append(data.get('lts_length', 1))

g_lts2_ig.add_edges(edges)
g_lts2_ig.es['lts_length'] = weights
print("igraph conversion complete.")


In [ ]:
## PREPARE GROUPED TRIPS FROM DEMAND
# We need to extract the start and end coordinates from the OD demand,
# then use OSMnx's nearest_nodes function to find the closest graph nodes,
# and finally build the grouped_trips dictionary in the format expected by the routing function.
# we do this for the bikeablity calculation function (speeds things up)
print("Extracting coordinates for vectorization...")
start_xs, start_ys = [], []
end_xs, end_ys = [], []
for geom in od_demand.geometry:
    coords = list(geom.coords)
    start_x, start_y = coords[0]
    end_x, end_y = coords[-1]
    start_xs.append(start_x)
    start_ys.append(start_y)
    end_xs.append(end_x)
    end_ys.append(end_y)

# sanity check
print(np.array(start_xs).ndim, np.array(start_xs).shape)  # should be (225,)

# vectorised nearest node lookup
u_nodes = ox.distance.nearest_nodes(G_base, X=start_xs, Y=start_ys)
v_nodes = ox.distance.nearest_nodes(G_base, X=end_xs,   Y=end_ys)

print("Building grouped_trips dictionary...")
grouped_trips = {}

for u, v, flow in zip(u_nodes, v_nodes, od_demand['total_flow']):
    if u in osmid_to_idx and v in osmid_to_idx:
        u_idx = osmid_to_idx[u]
        v_idx = osmid_to_idx[v]
        source_tuple = (u_idx,)
        if source_tuple not in grouped_trips:
            grouped_trips[source_tuple] = []
        grouped_trips[source_tuple].append({
            'ends': [v_idx],
            'flow': flow    })
print("Grouped trips ready!")

## load results

In [ ]:
# load
demand = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\gateshead\current_ltn_scenario\gateshead_poi_LTNs_tessellation_demand_weighted_current_ltn_scenario.pickle"
with open(demand, "rb") as f:
    demand_data = pickle.load(f)
    demand_gts = demand_data.get("GTs", [])
    demand_abs = demand_data.get("GT_abstracts", [])
    demand_gts.insert(0, nx.MultiGraph()) # add an empty one
    demand_abs.insert(0, nx.MultiGraph())
print("Demand Loaded")

betweeness = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\gateshead\current_ltn_scenario\gateshead_poi_LTNs_tessellation_betweenness_weighted_current_ltn_scenario.pickle"
with open(betweeness, "rb") as f:
    betweenness_data = pickle.load(f)
    betweenness_gts = betweenness_data.get("GTs", [])
    betweenness_abs = betweenness_data.get("GT_abstracts", [])
    betweenness_gts.insert(0, nx.MultiGraph())
    betweenness_abs.insert(0, nx.MultiGraph())
print("Betweeness Loaded")

betweeness_ltn_priority = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\gateshead\current_ltn_scenario\gateshead_poi_LTNs_tessellation_betweenness_ltn_priority_weighted_current_ltn_scenario.pickle"
with open(betweeness_ltn_priority, "rb") as f:
    betweenness_ltn_priority_data = pickle.load(f)
    betweenness_ltn_priority_gts = betweenness_ltn_priority_data.get("GTs", [])
    betweenness_ltn_priority_abs = betweenness_ltn_priority_data.get("GT_abstracts", [])
    betweenness_ltn_priority_gts.insert(0, nx.MultiGraph())
    betweenness_ltn_priority_abs.insert(0, nx.MultiGraph())
print("Betweeness LTN Loaded")

demand_ltn_priority = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\gateshead\current_ltn_scenario\gateshead_poi_LTNs_tessellation_demand_ltn_priority_weighted_current_ltn_scenario.pickle"
with open(demand_ltn_priority, "rb") as f:
    demand_ltn_priority_data = pickle.load(f)
    demand_ltn_priority_gts = demand_ltn_priority_data.get("GTs", [])
    demand_ltn_priority_abs = demand_ltn_priority_data.get("GT_abstracts", [])
    demand_ltn_priority_gts.insert(0, nx.MultiGraph())
    demand_ltn_priority_abs.insert(0, nx.MultiGraph())
print("Demand LTN Loaded")

random_runs = glob.glob(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\gateshead\current_ltn_scenario\gateshead_poi_LTNs_tessellation_random_weighted_current_ltn_scenario_run*.pickle")
random_gts_list = []
random_abs_list = []
for run in random_runs[:25]: # just 25 for speed
    with open(run, "rb") as f:
        random_data = pickle.load(f)
        r_gts = random_data.get("GTs", [])
        r_abs = random_data.get("GT_abstracts", [])
        r_gts.insert(0, nx.MultiGraph())
        r_abs.insert(0, nx.MultiGraph())
        random_gts_list.append(r_gts)
        random_abs_list.append(r_abs)
print("Random Loaded")

# trim to just 25 random for speed
random_gts_list = random_gts_list[:25]
random_abs_list = random_abs_list[:25]



## find bikeablity

In [ ]:
def get_bikeablity_lts_cutoff_igraph(results_gts, g_lts2_ig, grouped_trips, osmid_to_idx, cutoff=3000):
    bikeable_flows_per_stage = []
    for i, G_routed in enumerate(results_gts):
        g_current = g_lts2_ig.copy()
        if G_routed is not None and len(G_routed.edges) > 0:
            new_edges, new_weights = [], []
            for u, v, k, d in G_routed.edges(keys=True, data=True):
                if u in osmid_to_idx and v in osmid_to_idx:
                    new_edges.append((osmid_to_idx[u], osmid_to_idx[v]))
                    
                    raw_len = d.get('length', 0)
                    edge_len = float(raw_len if isinstance(raw_len, list) else raw_len)
                    new_weights.append(edge_len * 1) # Upgraded to LTS 1
                    
            g_current.add_edges(new_edges)
            
            start_idx = g_lts2_ig.ecount() 
            for idx, w in enumerate(new_weights):
                g_current.es[start_idx + idx]['lts_length'] = w
                
        stage_total_flow = 0
        for s_tuple, destinations in grouped_trips.items():
            unique_ends = list(set(e for d in destinations for e in d['ends']))
            if not unique_ends: continue
            
            # Dijkstra routing but only on the graph with a max lts of whatever we set the LTS_Limit to
            dists = np.array(g_current.distances(source=list(s_tuple), target=unique_ends, weights='lts_length'))
            min_dists = np.min(dists, axis=0)
            end_dist_map = dict(zip(unique_ends, min_dists))
            
            for d in destinations:
                trip_dist = min([end_dist_map[e] for e in d['ends']])
                if trip_dist <= cutoff:
                    stage_total_flow += d['flow']
                    
        bikeable_flows_per_stage.append(stage_total_flow)
    return bikeable_flows_per_stage


In [ ]:
# Calculate Bikeability for the Demand strategy
print("Calculating bikeability for Demand Weighted GTs...")
demand_bikeable_flows = get_bikeablity_lts_cutoff_igraph(
    results_gts=demand_gts, 
    g_lts2_ig=g_lts2_ig, 
    grouped_trips=grouped_trips, 
    osmid_to_idx=osmid_to_idx, 
    cutoff=3000
)

# Calculate Bikeability for the Random strategies
print("Calculating bikeability for Random GTs...")
random_bikeable_flows_list = []
for i, random_gts in enumerate(random_gts_list):
    flows = get_bikeablity_lts_cutoff_igraph(
        results_gts=random_gts, 
        g_lts2_ig=g_lts2_ig, 
        grouped_trips=grouped_trips, 
        osmid_to_idx=osmid_to_idx, 
        cutoff=3000
    )
    random_bikeable_flows_list.append(flows)

## make debug maps

In [ ]:
def generate_map(G_routed, G_abs, met_flow, stage, title, save_path, target_crs, target_demand_total):
    fig, ax = plt.subplots(figsize=(10, 10))
    
    lts_colors = {
            1: "green",
            2: "lightblue",
            3: "orange",
            4: "red"
        }

    # Map colours onto the GeoDataFrame
    G_base_edges_gdf["colour"] = G_base_edges_gdf["lts_class"].map(lts_colors)

    # Plot using the mapped colours
    G_base_edges_gdf.plot(
        ax=ax,
        color=G_base_edges_gdf["colour"],
        linewidth=0.8,
        alpha=0.6,
        legend=True,
        label="LTS Class",
        categorical=True)

    od_demand.plot(ax=ax, column='total_flow', cmap='cividis' ,legend=False, scheme="natural_breaks", markersize=50, alpha=0.7)

    # Add Labels 
    for _, row in od_demand.iterrows():
        if row.geometry is not None:
            point = row.geometry.interpolate(0.5, normalized=True)
            ax.text(
                point.x, point.y,
                f"{row['total_flow']:.1f}",
                fontsize=5, ha='center', va='center',
                bbox=dict(facecolor='white', alpha=0.1, edgecolor='none', pad=1),
                clip_on=True
            )

    edges_routed = None
    valid_routed = False
    if G_routed is not None and hasattr(G_routed, "edges") and len(G_routed.edges) > 0:
        if "crs" not in G_routed.graph:
            G_routed.graph["crs"] = target_crs
        _, edges_routed = ox.graph_to_gdfs(G_routed)
        edges_routed = edges_routed.to_crs(target_crs)
        edges_routed = edges_routed[
            edges_routed.geometry.notna() &
            ~edges_routed.geometry.is_empty]

        if not edges_routed.empty:
            bounds = edges_routed.total_bounds
            if np.all(np.isfinite(bounds)):
                
                if 'lts_class' not in edges_routed.columns:
                    # Convert base LTS to a dictionary for fast lookup
                    lts_lookup = G_base_edges_gdf['lts_class'].to_dict()
                    
                    lts_values = []
                    for u, v, k in edges_routed.index:
                        # 1. Check direct match (u, v)
                        if (u, v, k) in lts_lookup:
                            lts_values.append(lts_lookup[(u, v, k)])
                        # 2. Check reversed direction (v, u)
                        elif (v, u, k) in lts_lookup:
                            lts_values.append(lts_lookup[(v, u, k)])
                        # 3. Fallback (shouldn't really happen)
                        else:
                            lts_values.append(4) 
                            
                    # Assign the manually matched values back to the dataframe
                    edges_routed['lts_class'] = lts_values
                
                # Assign colours: Cyan if it's already LTS 1, Blue if it's a new upgrade
                edges_routed['route_colour'] = np.where(edges_routed['lts_class'] == 1, 'cyan', 'blue')
                
                # Plot using the new colour column
                edges_routed.plot(ax=ax, color=edges_routed['route_colour'], linewidth=2)
                
                valid_routed = True

    # if we were to plot abstract edges too, this is where we would do it
    # however the plot is already very busy!

    # Plot LTNs
    ltns.plot(ax=ax, alpha=0.5, color="orange")

    # Zoom
    if valid_routed and edges_routed is not None:
        minx, miny, maxx, maxy = edges_routed.total_bounds
    else:
        minx, miny, maxx, maxy = od_demand.total_bounds
        
    pad_x = (maxx - minx) * 0.30
    pad_y = (maxy - miny) * 0.30

    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)
    ax.set_autoscale_on(False)
    ax.set_axis_off()

    # Title
    pct_met = (met_flow / target_demand_total) * 100 if target_demand_total > 0 else 0
    plt.title(f"{title} | Stage {stage}\n Bikeable Trips: {met_flow:.1f} of {target_demand_total:.1f} ({pct_met:.1f}%)")
    
    plt.savefig(save_path, dpi=800, bbox_inches='tight')
    plt.close(fig)

In [ ]:
# Create output folders (Demand only for now, Random folders created dynamically)
base_output_path = r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\debugging\biketrips_debug"
output_dir_demand = os.path.join(base_output_path, "maps_demand")
os.makedirs(output_dir_demand, exist_ok=True)
target_crs = G_base_edges_gdf.crs
target_demand_total = od_demand['total_flow'].sum()

# Map demand
print("Generating Demand maps...")
for stage, (G_routed, G_abs, flow) in enumerate(zip(demand_gts, demand_abs, demand_bikeable_flows)):
    save_path = os.path.join(output_dir_demand, f"Demand_Stage_{stage}.png")
    generate_map(G_routed, G_abs, flow, stage, "Demand Weighted", save_path, target_crs, target_demand_total)
    print(f"Generated map for Demand Stage {stage}")

# Map random runs which beat demand
print("Evaluating all Random runs against Demand...")

# Loop through every single random run
for run_idx, (r_gts, r_abs, r_flows) in enumerate(zip(random_gts_list, random_abs_list, random_bikeable_flows_list)):
    output_dir_random = os.path.join(base_output_path, f"maps_random_run{run_idx}")
    # Track if we actually map anything in this run to avoid empty folders
    created_folder = False 
    # Loop through the stages within this specific random run
    for stage, (G_routed, G_abs, flow) in enumerate(zip(r_gts, r_abs, r_flows)):
        
        # Grab the benchmark demand flow for this exact stage
        demand_benchmark = demand_bikeable_flows[stage]
        
        #  Only plot if this random run beats the demand benchmark
        if flow > demand_benchmark:
            # Create the folder only when we find our first winner for this run
            if not created_folder:
                os.makedirs(output_dir_random, exist_ok=True)
                created_folder = True
                
            print(f"  Run {run_idx} Stage {stage} beat demand! ({flow:.1f} > {demand_benchmark:.1f}). Mapping it.")
            
            save_path = os.path.join(output_dir_random, f"Random_Run{run_idx}_Stage_{stage}.png")
            generate_map(G_routed, G_abs, flow, stage, f"Random Generation (Run {run_idx})", save_path, target_crs, target_demand_total)

print("All conditional mapping complete!")
print("Generating Debug Plot: Demand vs. Random Bikeability Scores...")

plt.figure(figsize=(12, 7))


stages = list(range(len(demand_bikeable_flows)))

# Plot every random run in light grey in the background
for i, r_flows in enumerate(random_bikeable_flows_list):
    # We only label the first one so the legend doesn't get 100 entries
    label = "Random Runs (Individual)" if i == 0 else None
    plt.plot(stages[:len(r_flows)], r_flows, color='grey', alpha=0.15, linewidth=1.5, label=label)

# Plot the Mean of the random runs (if they are all the same length)
try:
    random_mean = np.mean(random_bikeable_flows_list, axis=0)
    plt.plot(stages[:len(random_mean)], random_mean, color='blue', linestyle='--', linewidth=2.5, label='Random Runs (Mean)')
except Exception as e:
    print("Could not plot random mean (runs might have different stage lengths).")

# Plot the Demand baseline prominently in red
plt.plot(stages, demand_bikeable_flows, color='red', linewidth=3, label='Demand Benchmark')

plt.title("Diagnostics: Bikeability Score vs Iteration", fontsize=16, fontweight='bold')
plt.xlabel("Infrastructure Added (Stages / Iterations)", fontsize=14)
plt.ylabel("Bikeable Trips", fontsize=14)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right', fontsize=12)
plt.tight_layout()

debug_plot_path = os.path.join(base_output_path, "Diagnostics_Demand_vs_Random.png")
plt.savefig(debug_plot_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"Debug plot successfully saved to: {debug_plot_path}")